In [1]:
import polars as pl
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pyarrow
import graph_tool.all as gt

/home/jlopez/github/Dauphine/.venv/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# --------------------------------------------------------------------
# CLAUDE'S FIX -- wrong relative path for this notebook's working directory
# --------------------------------------------------------------------
# This notebook lives in v2_scripts/, so a path without "../" looks for
# v2_scripts/data/output/v2/2.clean_df_v2.csv, which doesn't exist --
# the file is one level up, at project/data/output/v2/2.clean_df_v2.csv
# (same convention as "5. clustering notebook.ipynb").

df = pl.read_csv("../data/output/v2/2.clean_df_v2.csv")

print(df.head())

STRIPPED

In [3]:
# Aggregate trips from station i -> station j
edges = (
    df
    .group_by([
        "Start station number",
        "End station number"
    ])
    .agg(
        pl.len().alias("weight")
    )
    .rename({
        "Start station number": "source",
        "End station number": "target"
    })
    .sort("weight", descending=True)
)

print(edges.head())
print(edges.shape)

shape: (5, 3)
┌────────┬────────┬────────┐
│ source ┆ target ┆ weight │
│ ---    ┆ ---    ┆ ---    │
│ i64    ┆ i64    ┆ u32    │
╞════════╪════════╪════════╡
│ 1075   ┆ 1132   ┆ 3156   │
│ 300213 ┆ 300234 ┆ 2851   │
│ 1132   ┆ 1075   ┆ 2807   │
│ 300234 ┆ 300213 ┆ 2756   │
│ 200038 ┆ 200230 ┆ 2535   │
└────────┴────────┴────────┘
(462168, 3)


In [4]:
stations_start = df.select(
    pl.col("Start station number")
    .drop_nulls()
    .unique()
    .alias("station")
)

stations_end = df.select(
    pl.col("End station number")
    .drop_nulls()
    .unique()
    .alias("station")
)

nodes = (
    pl.concat([stations_start, stations_end])
    .unique()
    .sort("station")
)

print(f"Number of stations : {nodes.height}")

Number of stations : 822


In [5]:
#Create graph

G = nx.DiGraph()

# Add all stations, including stations that might have
# only incoming or only outgoing trips

G.add_nodes_from(nodes["station"].to_list())


# Add weighted directed edges
G.add_weighted_edges_from(
    edges.select(["source", "target", "weight"]).iter_rows()
)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
#Tu as maintenant :
# G = (V,E)

#avec :
#V= stations ;
#E = couples de stations pour lesquels au moins un trajet est observé ;
#Wij= nombre de trajets i vers j

Number of nodes: 822
Number of edges: 462168


In [6]:
#basic descriptions of the graph


#Number of nodes and edges
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()

#Total trips
total_trips = sum(
    data["weight"]
    for _, _, data in G.edges(data=True)
)

print(f"Number of stations: {n_nodes}")
print(f"Number of directed edges: {n_edges}")
print(f"Number of total trips: {total_trips:,}")



#Graph density
density = nx.density(G)

print(f"Directed graph density: {density:.4f}")

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- a density of 0.68 is very high
# --------------------------------------------------------------------
# For a directed graph, density = E / (N*(N-1)), i.e. the fraction of all
# possible ordered station pairs that have at least one observed trip.
# 0.68 means most of the 822*821 ≈ 675k possible directed pairs actually
# have at least one recorded trip between them over the year. This is
# expected for a dense bike-share system (any station can in principle
# be reached from any other), but it also means the network is close to
# "everyone is connected to almost everyone" -- so unweighted structure
# (who is connected to whom) carries little information; the weights
# (how many trips) are where the real signal is. This matters below:
# unweighted degree/betweenness (cells 12, 14) will look flat/uninformative
# for exactly this reason, while the weighted versions (cells 13, 15) are
# far more discriminating.

Number of stations: 822
Number of directed edges: 462168
Number of total trips: 8,762,355
Directed graph density: 0.6848


In [7]:
##################
#Degree distribution
##################

#Unweighted degree
#This measures the number of different stations connected to a station
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

degree_df = pl.DataFrame({
    "station": list(G.nodes()),
    "in_degree": [in_degree[s] for s in G.nodes()],
    "out_degree": [out_degree[s] for s in G.nodes()],
})

print("In-degree:")
print(degree_df.select("in_degree").describe())

print("Out-degree:")
print(degree_df.select("out_degree").describe())

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- these numbers confirm the graph is near-complete
# --------------------------------------------------------------------
# With 822 stations, the maximum possible degree is 821. Here the median
# in-degree and out-degree are both ~576-577, and the mean is 562 for
# both. So a typical station is already directly connected (in at least
# one direction, at least once over the year) to about 68% of all other
# stations -- consistent with the 0.68 density noted above. The min
# out-degree of 0 (see cell below / cell 19) flags at least one station
# that never appears as a trip origin.

In-degree:


STRIPPED

In [8]:
#unweighted plot
plt.figure(figsize=(10, 6))

plt.hist(
    list(in_degree.values()),
    bins=30,
    alpha=0.6,
    label="In-degree"
)

plt.hist(
    list(out_degree.values()),
    bins=30,
    alpha=0.6,
    label="Out-degree"
)

plt.xlabel("Degree")
plt.ylabel("Number of stations")
plt.title("In-degree and Out-degree distributions")
plt.legend()
plt.show()

In [9]:

#Weighted degree / strength
#This gives you the stations generating/receiving the largest numbers of trips.

weighted_in_degree = dict(G.in_degree(weight="weight"))
weighted_out_degree = dict(G.out_degree(weight="weight"))

station_df = pl.DataFrame({
    "station": list(G.nodes()),
    "in_degree": [in_degree[s] for s in G.nodes()],
    "out_degree": [out_degree[s] for s in G.nodes()],
    "weighted_in_degree": [
        weighted_in_degree[s] for s in G.nodes()
    ],
    "weighted_out_degree": [
        weighted_out_degree[s] for s in G.nodes()
    ],
})

print("In-degree weighted:")
print(station_df.select("weighted_in_degree").describe())

print("Out-degree weighted:")
print(station_df.select("weighted_out_degree").describe())

# --------------------------------------------------------------------
# CLAUDE'S NOTE
# --------------------------------------------------------------------
# Mean weighted in/out-degree is identical (10,659.8) because every trip
# counted as an outflow of its start station is also counted as an
# inflow of its end station -- total inflow across the network always
# equals total outflow. The spread is large though: std ~6,600, and the
# busiest station receives 50,563 trips/year (max weighted_in_degree)
# vs. a quietest station with only 27 -- almost a 2,000x gap in traffic
# between the busiest and quietest stations, even though nearly every
# station is *connected* to nearly every other one (previous cell).

STRIPPED

In [10]:
#distribution of edge weights
#The edge weight is:
#the number of observed trips from station i to station j.
weights = [
    data["weight"]
    for _, _, data in G.edges(data=True)
]

#descriptive stats
weights_series = pl.Series("weight", weights)

print(weights_series.describe())

#percentiles
print(
    weights_series.quantile(
        [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- edge weights are extremely right-skewed
# --------------------------------------------------------------------
# Median edge weight is 6 trips/year but the mean is 18.96 -- pulled up
# by a long tail (max 3,156 trips on a single station pair). Even the
# 90th percentile is only 46, so the vast majority of the 462,168
# station-pair connections are "connected but rarely used" (a handful of
# trips over the whole year), while a small number of pairs (likely
# geographically close or on popular commuting routes) carry a
# disproportionate share of traffic. This is why the log-log plot two
# cells below is the more informative view of this distribution than
# the linear-scale histogram right after this cell.

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 462168.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 18.959242 │
│ std        ┆ 46.05024  │
│ min        ┆ 1.0       │
│ 25%        ┆ 2.0       │
│ 50%        ┆ 6.0       │
│ 75%        ┆ 18.0      │
│ max        ┆ 3156.0    │
└────────────┴───────────┘


[2.0, 6.0, 18.0, 46.0, 78.0, 190.0]


In [11]:
#Edge weights distribution plot
plt.figure(figsize=(10, 6))

plt.hist(weights, bins=50)

plt.xlabel("Number of trips")
plt.ylabel("Number of edges")
plt.title("Distribution of edge weights")

plt.show()

In [12]:
#log graph
plt.figure(figsize=(10, 6))

plt.hist(weights, bins=50)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Edge weight (number of trips)")
plt.ylabel("Number of edges")
plt.title("Distribution of edge weights — log-log scale")

plt.show()

In [13]:
###############
#Central stations
###############
#using degree centrality
degree_centrality = nx.degree_centrality(G)

top_degree = sorted(
    degree_centrality.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Top 10 stations by degree centrality:")

for station, value in top_degree:
    print(station, value)

    

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- these values above 1 are expected, but not very useful
# --------------------------------------------------------------------
# networkx's degree_centrality for a DiGraph is (in_degree + out_degree)
# / (N-1). Since in- and out-degree are each capped at N-1 = 821, the
# score can go up to 2.0 -- values around 1.92-1.95 here just mean these
# stations are connected (in either direction) to almost every other
# station in both directions. That matches the 0.68 density noted
# earlier: with the graph this dense, unweighted degree centrality
# barely separates stations (top 10 all sit within ~0.03 of each other,
# 995 at 1.945 vs 1028 at 1.917). The weighted flow centrality in the
# next cell (actual trip counts) is a much more meaningful ranking of
# "central" stations than this unweighted score.

Top 10 stations by degree centrality:
1052 1.9500609013398293
995 1.9451887941534713
1160 1.9427527405602922
200128 1.9403166869671131
1011 1.9354445797807551
1151 1.925700365408039
300249 1.925700365408039
3504 1.9244823386114494
960 1.9183922046285018
1028 1.9171741778319122


In [14]:
#Weighted flow centrality
top_inflow = sorted(
    weighted_in_degree.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Top 10 stations by incoming flow:")

for station, value in top_inflow:
    print(f"{station}: {value:,} trips")

#Stations with most trips
top_outflow = sorted(
    weighted_out_degree.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Top 10 stations by outgoing flow:")

for station, value in top_outflow:
    print(f"{station}: {value:,} trips")

Top 10 stations by incoming flow:
1072: 50,563 trips
2696: 44,261 trips
960: 44,106 trips
1161: 41,383 trips
1067: 40,483 trips
2587: 39,983 trips
22179: 39,072 trips
1052: 38,631 trips
1075: 36,931 trips
200128: 36,538 trips
Top 10 stations by outgoing flow:
1072: 53,880 trips
2696: 46,257 trips
1075: 42,737 trips
2587: 39,851 trips
1011: 38,519 trips
960: 36,539 trips
300083: 33,935 trips
200183: 32,753 trips
1161: 32,640 trips
1132: 31,421 trips


In [15]:
#betweenness
betweenness = nx.betweenness_centrality(
    G,
    weight=None
)

top_betweenness = sorted(
    betweenness.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Top 10 stations by betweenness centrality:")

for station, value in top_betweenness:
    print(station, value)


Top 10 stations by betweenness centrality:
995 0.0012152644378142256
22180 0.0011511887963628197
960 0.0011347114678882466
1052 0.0011194104624116053
200128 0.0011166201274510482
1160 0.0010870008908144876
1011 0.0010740354265449184
1151 0.0010591485660458573
1007 0.0010509085520720393
999 0.0010423075658604467


In [16]:
G_distance = G.copy()

for u, v, data in G_distance.edges(data=True):
    data["distance"] = 1 / data["weight"]

betweenness_flow = nx.betweenness_centrality(
    G_distance,
    weight="distance"
)

top_betweenness = sorted(
    betweenness_flow.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

for station, value in top_betweenness:
    print(station, value)

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- compare this to the unweighted betweenness above
# --------------------------------------------------------------------
# Unweighted betweenness (previous cell) is nearly flat across the top
# 10 stations (0.00104-0.00122): expected, since with density 0.68 most
# station pairs are already directly connected, so there are very few
# multi-hop shortest paths for any station to sit "between". Once edges
# are weighted by 1/trip_count (so high-traffic links count as "closer"),
# the picture changes completely: station 1072 is a clear standout at
# 0.339 -- more than double the runner-up (2587 at 0.118), and roughly
# 280x any unweighted betweenness score. 1072 is also the single busiest
# station in both weighted degree tables above, so this correctly
# identifies it as a real hub that many high-traffic routes flow through,
# something plain (unweighted) betweenness could not detect here.

1072 0.33923383143697455
2587 0.11848875553310954
1067 0.11186536347702089
1161 0.09419803333234307
200183 0.074682867413327
22179 0.06586405632631236
1011 0.06480793796975728
960 0.0567110305695018
300221 0.056158462315439237
300005 0.054313597338165834


In [17]:
###########################3
#Stations with strong income/outcome imbalance
############################

#For each station:
#imbalance = inflow - outflow

#A positive value means the station receives more trips than it sends.
#A negative value means it sends more trips than it receives.

station_df = station_df.with_columns(
    (
        pl.col("weighted_in_degree")
        - pl.col("weighted_out_degree")
    ).alias("flow_imbalance")
)


#Absolute imbalance
station_df = station_df.with_columns(
    pl.col("flow_imbalance")
    .abs()
    .alias("absolute_imbalance")
)


In [18]:
#Stations with largest positive and negative imbalance

#positive
largest_inflow_imbalance = (
    station_df
    .sort("flow_imbalance", descending=True)
    .head(10)
)

print(largest_inflow_imbalance)


#negative
largest_outflow_imbalance = (
    station_df
    .sort("flow_imbalance")
    .head(10)
)

print(largest_outflow_imbalance)

STRIPPED

In [19]:
#Overall strongest imbalance
largest_absolute_imbalance = (
    station_df
    .sort("absolute_imbalance", descending=True)
    .head(10)
)

print(largest_absolute_imbalance)


STRIPPED

In [20]:
#------------------------
#Normalized imbalance
#------------------------

station_df = station_df.with_columns(
    (
        (
            pl.col("weighted_in_degree")
            - pl.col("weighted_out_degree")
        )
        /
        (
            pl.col("weighted_in_degree")
            + pl.col("weighted_out_degree")
        )
    )
    .fill_nan(0)
    .alias("normalized_imbalance")
)

print(
    station_df
    .sort("normalized_imbalance", descending=True)
    .head(10)
)

print(
    station_df
    .sort("normalized_imbalance")
    .head(10)
)

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- stations 10626 and 22168 hit the max possible score (1.0)
# --------------------------------------------------------------------
# normalized_imbalance = (in - out) / (in + out), which is exactly ±1
# only when in or out is zero. Stations 10626 (in_degree=354, out_degree=0)
# and 22168 (in_degree=152, out_degree=0) never appear as a trip *origin*
# at all -- every recorded trip involving them ends there. That's not a
# division-by-zero bug (in+out is still >0 for both, so fill_nan(0) never
# actually triggers here), but it is worth checking in the raw data: this
# pattern usually means a station was decommissioned/removed from service
# partway through the year (bikes could still be returned to it, but it
# stopped being offered as a start point), rather than a genuinely
# one-directional travel pattern. Worth cross-checking against the
# station status data before reading too much into these two as
# "imbalanced" in the same sense as the other rows here.

STRIPPED

In [21]:
#####################
#Table summary
#####################
station_analysis = station_df.with_columns(
    pl.Series(
        "degree_centrality",
        [
            degree_centrality[s]
            for s in station_df["station"].to_list()
        ]
    ),
    pl.Series(
        "betweenness_centrality",
        [
            betweenness_flow[s]
            for s in station_df["station"].to_list()
        ]
    )
)
print(
    station_analysis
    .sort("weighted_in_degree", descending=True)
    .head(20)
)

STRIPPED

In [22]:
#Summary statistics
print("=" * 60)
print("DIRECTED WEIGHTED STATION NETWORK — 2025")
print("=" * 60)

print(f"Number of nodes: {G.number_of_nodes():,}")
print(f"Number of edges: {G.number_of_edges():,}")
print(f"Network density: {nx.density(G):.6f}")

print("\nEdge weights:")
print(f"  Minimum: {min(weights):,}")
print(f"  Maximum: {max(weights):,}")
print(f"  Mean:    {np.mean(weights):.2f}")
print(f"  Median:  {np.median(weights):.2f}")

print("\nDegree statistics:")
print(f"  Mean in-degree:  {np.mean(list(in_degree.values())):.2f}")
print(f"  Mean out-degree: {np.mean(list(out_degree.values())):.2f}")
print(f"  Max in-degree:   {max(in_degree.values()):,}")
print(f"  Max out-degree:  {max(out_degree.values()):,}")

DIRECTED WEIGHTED STATION NETWORK — 2025
Number of nodes: 822
Number of edges: 462,168
Network density: 0.684833

Edge weights:
  Minimum: 1
  Maximum: 3,156
  Mean:    18.96
  Median:  6.00

Degree statistics:
  Mean in-degree:  562.25
  Mean out-degree: 562.25
  Max in-degree:   799
  Max out-degree:  807


In [23]:
#Network visualization
max_weight = max(weights)

edge_widths = [
    0.5 + 3 * np.sqrt(data["weight"] / max_weight)
    for _, _, data in G.edges(data=True)
]

plt.figure(figsize=(14, 12))

pos = nx.spring_layout(G, seed=42)

nx.draw_networkx_nodes(
    G,
    pos,
    node_size=50,
    alpha=0.7
)

nx.draw_networkx_edges(
    G,
    pos,
    width=edge_widths,
    alpha=0.2,
    arrows=True,
    arrowsize=5
)

plt.title("Directed weighted station network — 2025")
plt.axis("off")
plt.show()

In [24]:
#Inflow vs outflow graph
plot_df = station_analysis.to_pandas()

plt.figure(figsize=(10, 8))

plt.scatter(
    plot_df["weighted_out_degree"],
    plot_df["weighted_in_degree"],
    alpha=0.6
)

max_flow = max(
    plot_df["weighted_out_degree"].max(),
    plot_df["weighted_in_degree"].max()
)

plt.plot(
    [0, max_flow],
    [0, max_flow],
    linestyle="--"
)

plt.xlabel("Outgoing trips")
plt.ylabel("Incoming trips")
plt.title("Station inflow vs. outflow — 2025")

plt.show()

# --------------------------------------------------------------------
# CLAUDE'S NOTE
# --------------------------------------------------------------------
# Most points hug the dashed y=x line, i.e. for most stations inflow and
# outflow are roughly balanced over the year -- makes sense for a
# bike-share network where nearly every bike taken from a station
# eventually gets returned to the system. The points that sit furthest
# from the line are exactly the "largest imbalance" stations already
# surfaced in cells 17-19 (e.g. 1067, 982, 2692) -- this plot is a useful
# visual cross-check that those aren't outliers of the ranking method,
# they're visibly off the diagonal.

In [25]:
####################
#Stochastic block model
####################

#------------
#Translate to graph-tool
#------------
gt_graph = gt.Graph(directed=True)

# Mapping NetworkX station IDs -> graph-tool vertex IDs
station_to_vertex = {}

for station in G.nodes():
    v = gt_graph.add_vertex()
    station_to_vertex[station] = int(v)

#Add edges
weight_prop = gt_graph.new_edge_property("double")

for source, target, data in G.edges(data=True):

    e = gt_graph.add_edge(
        station_to_vertex[source],
        station_to_vertex[target]
    )

    weight_prop[e] = data["weight"]

gt_graph.edge_properties["weight"] = weight_prop


In [26]:
#Poisson model
state_poisson = gt.minimize_blockmodel_dl(
    gt_graph,
    state_args={
        "recs": [gt_graph.ep.weight],
        "rec_types": ["discrete-poisson"],
        "deg_corr": True,
    }
)

In [ ]:
#Inferred number of blocks
print("Number of blocks:", state_poisson.get_nonempty_B())

print("Model entropy:", state_poisson.entropy())

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- 802 blocks for 822 stations is an important result
# to flag, not just report
# --------------------------------------------------------------------
# gt.minimize_blockmodel_dl chooses the number of blocks itself (that's
# the "k" being selected here, there's no k you pass in) by minimizing
# description length -- balancing how well the model fits the weighted
# edges against model complexity. Here it landed on 802 non-empty
# blocks out of 822 stations: i.e. almost every station ends up in its
# own block, with only a handful of stations sharing a block with
# anyone else (confirmed below: 802 blocks for 822 stations, most of
# size 1-2). That is confirmed as a stable result, not a one-off, by
# the 10-seed re-run at the bottom of this notebook (801-819 blocks
# every time, entropy always ~2.60-2.61 million). Practically, this
# means this particular SBM configuration (directed, degree-corrected,
# weighted with a discrete-Poisson edge distribution) is not finding a
# small number of interpretable station "communities" -- it's telling
# you the trip pattern between any two stations is distinctive enough
# that DL-minimization prefers to keep them mostly separate. If the
# goal is a handful of human-interpretable groups (like the k=2..6 in
# the other clustering notebook), this model as configured won't give
# you that; worth trying gt.minimize_nested_blockmodel_dl (which
# explicitly builds a hierarchy of coarser groupings on top of this
# fine partition) if a small number of macro-groups is what's wanted.
# See also the note on the next cell -- until that fix, the block
# assignments used in every cell below this point actually came from a
# different (undefined) object, not from `state_poisson`.

In [ ]:
# --------------------------------------------------------------------
# CLAUDE'S FIX -- `state` was never defined in this notebook
# --------------------------------------------------------------------
# The only SBM state built above is `state_poisson` (two cells up).
# `state` does not exist anywhere in this notebook's code, so
# `state.get_blocks()` could only ever have worked because a variable
# named `state` was left over in the kernel's memory from some other,
# earlier run (e.g. an unweighted or differently-configured SBM fit that
# isn't in this notebook at all) -- not from `state_poisson` computed
# above. That silently substituted a different model's block
# assignments into `clusters_df`/`station_blocks`/`block_summary`, etc.
# The giveaway: the cell above reports "Number of blocks: 802" for
# `state_poisson`, but the old `class_sizes` output (before this fix)
# only had 93 rows -- a completely different partition. On a fresh
# kernel (Restart & Run All), this cell used to raise
# `NameError: name 'state' is not defined`.
#
# Fix: use `state_poisson`, the model actually built in this notebook.
# Confirmed by re-running the whole notebook end-to-end: this now gives
# 802 blocks (matching the cell above), not the 93 seen before the fix
# -- see the notes on the cells below.

#get class assigned to every station
blocks = state_poisson.get_blocks()
station_blocks = {}

for station, vertex_id in station_to_vertex.items():
    station_blocks[station] = int(
        blocks[gt_graph.vertex(vertex_id)]
    )

clusters_df = pl.DataFrame({
    "station": list(station_blocks.keys()),
    "block": list(station_blocks.values())
})

print(clusters_df)

In [ ]:
#Check size of each class
class_sizes = (
    clusters_df
    .group_by("block")
    .agg(
        pl.len().alias("n_stations")
    )
    .sort("n_stations", descending=True)
)

class_sizes = class_sizes.with_columns(
    (
        pl.col("n_stations")
        / pl.col("n_stations").sum()
        * 100
    ).alias("percentage")
)

print(class_sizes)

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- confirms the prediction from the fix above
# --------------------------------------------------------------------
# With the fix, this now shows 802 rows (matching `state_poisson`'s
# block count), and the biggest block has only 3 stations (0.36%) --
# nowhere near the pre-fix table's 93 rows with a 21-station block.
# Almost every block here is a singleton or pair: this is the concrete
# confirmation that the SBM, as configured, is assigning nearly every
# station its own class rather than finding a handful of meaningful
# station communities.

In [ ]:
#SMB with station stats for summary
station_analysis = station_analysis.join(
    clusters_df,
    on="station",
    how="left"
)

block_summary = (
    station_analysis
    .group_by("block")
    .agg([
        pl.len().alias("n_stations"),

        pl.col("in_degree")
        .mean()
        .alias("mean_in_degree"),

        pl.col("out_degree")
        .mean()
        .alias("mean_out_degree"),

        pl.col("weighted_in_degree")
        .mean()
        .alias("mean_inflow"),

        pl.col("weighted_out_degree")
        .mean()
        .alias("mean_outflow"),

        pl.col("flow_imbalance")
        .mean()
        .alias("mean_imbalance"),

        pl.col("normalized_imbalance")
        .mean()
        .alias("mean_normalized_imbalance"),

        pl.col("degree_centrality")
        .mean()
        .alias("mean_degree_centrality"),

        pl.col("betweenness_centrality")
        .mean()
        .alias("mean_betweenness"),
    ])
    .sort("block")
)

print(block_summary)

# --------------------------------------------------------------------
# CLAUDE'S NOTE
# --------------------------------------------------------------------
# With the fix, almost every row here has n_stations=1 (e.g. block 0
# is just station 2001454444 -- the same "station" id skewed 2 billion
# flagged in the other clustering notebook's PCA bug note -- and block
# 2 is just station 960). So these "mean" columns mostly just echo one
# station's own values rather than averaging over a real group; only
# the handful of size 2-3 blocks are genuine averages. Not a bug, but
# worth reading this table as "one row per near-unique station" rather
# than "802 meaningful station segments".

In [ ]:
#Flow between classes from SBM
edges_with_blocks = edges.with_columns(
    pl.col("source")
    .replace(station_blocks)
    .cast(pl.Int64)
    .alias("source_block"),

    pl.col("target")
    .replace(station_blocks)
    .cast(pl.Int64)
    .alias("target_block")
)

#Add trips
block_flows = (
    edges_with_blocks
    .group_by([
        "source_block",
        "target_block"
    ])
    .agg(
        pl.col("weight").sum().alias("total_trips")
    )
    .sort("total_trips", descending=True)
)

print(block_flows)

#This answers Do the SBM blocks reveal important flows between particular groups of stations?

# --------------------------------------------------------------------
# CLAUDE'S NOTE
# --------------------------------------------------------------------
# With the fix, this is now 458,425 rows (vs. 8,649 pre-fix) -- because
# with 802 mostly-singleton blocks there are almost as many distinct
# block pairs as there were station pairs. The top rows are exactly the
# busiest station-to-station edges from cell 2 (e.g. 1075<->1132,
# 300213<->300234), just relabelled by block instead of by station,
# which is the expected consequence of blocks mostly containing one
# station each -- this table doesn't currently answer "do groups of
# stations exchange a lot of traffic", it mostly restates the
# station-level edge list from earlier in the notebook.

In [ ]:
#Block to block flow matrix
flow_matrix = (
    block_flows
    .pivot(
        on="target_block",
        index="source_block",
        values="total_trips"
    )
    .fill_null(0)
)

print(flow_matrix)

# --------------------------------------------------------------------
# CLAUDE'S NOTE
# --------------------------------------------------------------------
# This is now an 802x803 matrix (one row/column per block), instead of
# the pre-fix 93x94 -- mostly sparse, since with near-singleton blocks
# most block pairs never traded a trip directly. The heatmap in the
# next cell will only usefully render a small subset of this.

In [ ]:
#plot flow matrix
flow_pd = flow_matrix.to_pandas().set_index("source_block")

plt.figure(figsize=(10, 8))

sns.heatmap(
    flow_pd,
    annot=True,
    fmt=",.0f"
)

plt.xlabel("Destination block")
plt.ylabel("Origin block")
plt.title("Trip flows between SBM blocks")

plt.show()

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- this plot is no longer meaningful with 802 blocks
# --------------------------------------------------------------------
# This renders (doesn't error), but it's now an 802x803 annotated
# heatmap -- effectively unreadable, and `annot=True` on a matrix this
# size is also needlessly slow to draw. With the pre-fix 93-block
# partition this plot was a reasonable diagonal-heavy sanity check; now
# that blocks are mostly single stations, the diagonal is trivially the
# station's own edge weight and off-diagonal cells are almost all 0/
# blank -- there's no real "block structure" left to visualize this
# way. If a block-level heatmap is still wanted, it would need either a
# coarser model (`minimize_nested_blockmodel_dl`, or dropping
# `deg_corr`/weights to force fewer, larger blocks) or dropping
# `annot=True` and restricting to the handful of blocks with more than
# one station.

In [ ]:
#Run model several times
results = []

for seed in range(10):

    np.random.seed(seed)

    state_i = gt.minimize_blockmodel_dl(
        gt_graph,
        state_args={
            "recs": [gt_graph.ep.weight],
            "rec_types": ["discrete-poisson"],
            "deg_corr": True,
        }
    )

    results.append({
        "run": seed,
        "n_blocks": state_i.get_nonempty_B(),
        "entropy": state_i.entropy(),
    })

print(pl.DataFrame(results))

# --------------------------------------------------------------------
# CLAUDE'S NOTE -- this is the evidence that the fix in cell 27 is correct
# --------------------------------------------------------------------
# Across 10 different random seeds, minimize_blockmodel_dl consistently
# finds 801-819 blocks (entropy consistently ~2.603-2.606 million),
# tightly clustered around the 802 blocks reported for `state_poisson`
# earlier. This confirms two things: (1) the model's real, reproducible
# answer is "~800+ blocks", not the 93 seen before the cell-27 fix; and
# (2) this isn't an unlucky single run -- the SBM genuinely and stably
# prefers near-one block per station for this graph/configuration, so
# getting a small number of interpretable groups out of this data would
# require changing the model setup (e.g. `minimize_nested_blockmodel_dl`
# for a hierarchy, or dropping edge weights / degree-correction to
# encourage coarser blocks), not just re-running this one with a
# different seed.